In [2]:
!uv pip install torch



Checked 1 package in 408ms


In [11]:
import os
import re
from openai import OpenAI
import pymupdf
import numpy as np
import fitz
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [5]:
doc = fitz.open(r"E:\AI-ML\MOF_Economic Survey ENG 2023-24 book Final_for WEB_miwtpw0.pdf")

In [31]:
text = ""

for page in doc:
    text += page.get_text() + "\n"


# =========================================================
# STEP 1: FIRST CHUNKING
# =========================================================

chunk_size = 1000
overlap = 100

first_chunks = []

start = 0

while start < len(text):
    end = start + chunk_size

    first_chunks.append(text[start:end])

    start += chunk_size - overlap

print("First-level chunks:", len(first_chunks))


# =========================================================
# STEP 2: SECOND CHUNKING
# =========================================================

second_chunk_size = 400
second_overlap = 50

chunks = []

for big_chunk in first_chunks:

    start = 0

    while start < len(big_chunk):

        end = start + second_chunk_size

        small_chunk = big_chunk[start:end]

        if small_chunk.strip():
            chunks.append(small_chunk)

        start += second_chunk_size - second_overlap


print("Final chunks:", len(chunks))


# =========================================================
# STEP 3: VIEW FINAL CHUNKS
# =========================================================

for i, chunk in enumerate(chunks, start=1):

    print(f"\n{'='*60}")
    print(f"Chunk {i}")
    print(f"{'='*60}")

    print(chunk)

First-level chunks: 1141
Final chunks: 3423

Chunk 1
Economic Survey 2023/24
Government of Nepal
Ministry of Finance
Singh Durbar, Kathmandu
-4.0
-2.0
0.0
2.0
4.0
6.0
8.0
10.0
12.0
Economic Growth Rate
Consumer Price Inflation
Economic Growth Rate and Consumer Price Inflation (In Percent)
0
500
1000
1500
2000
2014/152015/162016/172017/182018/192019/202020/212021/222022/23
Remittance Inflow
Foreign Exchange Reserve
Remittance Inflow and Foreign Excha

Chunk 2
change Reserve
Remittance Inflow and Foreign Exchange Reserve (Rs.In Billion)
Koshi
16%
Madhes
13%
Bagmati
37%
Gandaki
9%
Lumbini
14%
Karnali
4%
Sudurpaschim
7%
Province-wise GDP Composition
 
0.0
5.0
10.0
15.0
20.0
25.0
30.0
35.0
2014/152015/162016/172017/182018/192019/202020/212021/222022/23
Revenue/GDP
Expenditure/GDP
Trend of Revenue and Expenditure (In Percent)
0.0
5.0
10.0
15.0
20.0
25.0
30.

Chunk 3
iture (In Percent)
0.0
5.0
10.0
15.0
20.0
25.0
30.0
Deposit Growth Rate
Private Sector Credit Growth Rate
Deposit and Private S

In [33]:
embedder = make_pipeline(
    TfidfVectorizer(
        stop_words="english",
        max_features=10000
    ),
    TruncatedSVD(
        n_components=20,
        random_state=0
    ),
    Normalizer()
)

index = embedder.fit_transform(chunks)

def retrieve(question, k=100):
    q = embedder.transform([question])[0]
    scores = index @ q
    return [
        (scores[i], chunks[i])
        for i in np.argsort(-scores)[:k]
    ]

In [34]:
retrieve("what is the rate of remittance")

[(np.float64(0.9290138149651835),
  'titution\n1116\n1378\n1895\n2449\n3629\n4057\n4685\n5134\n5128\n5130\nPopulation per Bank Branch\n7206\n6562\n5610\n4334\n3363\n3072\n2844\n2532\n2517\n2516\nNumber of Deposits Accounts\n14934618 16836017 19677005 23544859 27866505 32454204 37770985 44971969 51177923 54400720\nNumber of Loan Accounts\n1033383\n1096570\n1216091\n1301010\n1439648\n1544059\n1702195\n1829044\n1845213\n1876473\nBranchless Banking Center\n504\n812\n1008\n128'),
 (np.float64(0.917319185782486),
  '1567.1\n56141.3\nB\nLiquidity Absorption\n47680.0\n54255.0\n12445.0\n19500.0\n10035.0\n7800.0\n30329.0\n6000.0\n10820.0\n500.0\n115985.0\nReverse Repo Auction\n31580.0\n23595.0\n6425.0\n8475.0\n2070.0\n4800.0\n10954.0\n2835.0\n8820.0\n500.0\n0.0\nOutright Sale  Auction\n600.0\n910.0\n0.0\n840.0\n0.0\n0.0\n0.0\n0.0\n0.0\n0.0\n0.0\nDeposit Collection Auctions\n15500\n29750\n1645\n5590\n7965\n3000\n19375\n3165\n2000\n0\n59305\nDeposit Collection \nAuc'),
 (np.float64(0.9161086875840

In [35]:
context = retrieve("what is the rate of remittance")

In [36]:
import os
from openai import OpenAI

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6Llb0dc6kUs6tekWVogpGo84Cu2Uos0qP9ujGJg_j5qnw"

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

system_prompt = f"""
you are FAQ of government data.You have to clearly show the having data and also clearify to user.
you make your response more short and clear for user friendly.
"""

user_query = """
what is the rate of remittance
"""

response = client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_query
        }
    ]
)

In [39]:
ai_response = response.choices[0].message.content
ai_response

'**FAQ: Remittance Rates**\n\n**Data Availability:**\nWe do not hold a single, fixed "rate of remittance" because remittance rates vary constantly. \n\n**What depends on the rate?**\n*   **Transfer Method:** Banks, money transfer operators (e.g., Western Union, Wise), and crypto/digital wallets.\n*   **Source & Destination Countries:** The corridors through which money is sent.\n*   **Currency Exchange Margins:** Daily fluctuating foreign exchange rates.\n*   **Service Fees:** Flat fees or percentage-based charges set by the specific financial provider.\n\n**Need specific data?**\nTo get the exact current rate, please specify:\n1. **The sending country** \n2. **The receiving country**'